# Comprehensive EDA, Training and Experiments Notebook

This notebook provides a complete pipeline for:
- Exploratory Data Analysis (EDA)
- Data Preprocessing and Feature Engineering
- Model Training and Evaluation
- Hyperparameter Tuning
- Experiment Tracking and Results Comparison

Let's begin with a systematic approach to machine learning model development.

## 1. Import Required Libraries

Import essential libraries for data analysis, visualization, and machine learning.

In [1]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
from scipy import stats

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, roc_curve, auc, roc_auc_score)

# Utilities
import warnings
import json
from datetime import datetime
import pickle
import os

warnings.filterwarnings('ignore')

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 2. Load and Explore Dataset

Load the dataset and perform initial exploration to understand its structure and characteristics.

In [ ]:
# Create sample dataset for demonstration (replace with your actual data)
# For production, use: df = pd.read_csv('your_dataset.csv')

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

# Load example dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

# Display basic information
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"   - Rows: {df.shape[0]}")
print(f"   - Columns: {df.shape[1]}")

# Display first few rows
print(f"\n📋 First 5 Rows:")
print(df.head())

# Display data types
print(f"\n🔍 Data Types:")
print(df.dtypes)

# Display missing values
print(f"\n❌ Missing Values:")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("   No missing values found!")
else:
    print(missing_values[missing_values > 0])

# Display basic statistics
print(f"\n📈 Statistical Summary:")
print(df.describe())

## 3. Data Preprocessing and Cleaning

Handle missing values, remove duplicates, and correct data types.

In [ ]:
print("=" * 80)
print("DATA PREPROCESSING AND CLEANING")
print("=" * 80)

# Create a copy for processing
df_clean = df.copy()

# Check for duplicates
print(f"\n🔄 Duplicate Rows: {df_clean.duplicated().sum()}")
if df_clean.duplicated().sum() > 0:
    df_clean = df_clean.drop_duplicates()
    print("   ✓ Duplicates removed!")

# Handle missing values
print(f"\n❌ Missing Values Summary:")
missing_count = df_clean.isnull().sum()
if missing_count.sum() == 0:
    print("   No missing values detected!")
else:
    for col in df_clean.columns:
        if df_clean[col].isnull().sum() > 0:
            print(f"   {col}: {df_clean[col].isnull().sum()} missing values")
            # Fill missing values with median for numerical columns
            if df_clean[col].dtype in ['int64', 'float64']:
                df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Check and correct data types
print(f"\n📝 Data Types Verification:")
print(df_clean.dtypes)

# Identify numerical and categorical columns
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 Column Classification:")
print(f"   Numerical Columns: {len(numerical_cols)} - {numerical_cols[:5]}{'...' if len(numerical_cols) > 5 else ''}")
print(f"   Categorical Columns: {len(categorical_cols)} - {categorical_cols}")

print(f"\n✓ Data Preprocessing Complete!")
print(f"   Cleaned Dataset Shape: {df_clean.shape}")

## 4. Exploratory Data Analysis (EDA)

Generate statistical summaries and create comprehensive visualizations.

In [ ]:
print("=" * 80)
print("EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 80)

# Separate features and target
X = df_clean.drop('target', axis=1)
y = df_clean['target']

print(f"\n📊 Target Variable Distribution:")
print(y.value_counts())
print(f"\nTarget Class Proportions:")
print(y.value_counts(normalize=True).round(4))

# Create comprehensive visualizations
fig = plt.figure(figsize=(20, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# 1. Target Distribution
ax1 = fig.add_subplot(gs[0, 0])
y.value_counts().plot(kind='bar', ax=ax1, color=['#FF6B6B', '#4ECDC4'])
ax1.set_title('Target Variable Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count')
ax1.set_xlabel('Class')

# 2. Target Distribution (Pie Chart)
ax2 = fig.add_subplot(gs[0, 1])
y.value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4'])
ax2.set_title('Target Class Proportion', fontsize=12, fontweight='bold')
ax2.set_ylabel('')

# 3. Feature Statistics
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
stats_text = f"""
DATASET STATISTICS
─────────────────
Total Samples: {X.shape[0]}
Total Features: {X.shape[1]}
Missing Values: {X.isnull().sum().sum()}

FEATURE RANGES
─────────────────
Min Value: {X.min().min():.4f}
Max Value: {X.max().max():.4f}
Mean Value: {X.mean().mean():.4f}
Std Dev: {X.std().mean():.4f}
"""
ax3.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 4. Histograms of first 3 features
for idx, col in enumerate(X.columns[:3]):
    ax = fig.add_subplot(gs[1, idx])
    ax.hist(X[col], bins=30, alpha=0.7, color='#4ECDC4', edgecolor='black')
    ax.set_title(f'Distribution: {col.split("(")[0][:20]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

# 5. Box plots for selected features
for idx, col in enumerate(X.columns[:3]):
    ax = fig.add_subplot(gs[2, idx])
    sns.boxplot(data=df_clean, x='target', y=col, ax=ax, palette=['#FF6B6B', '#4ECDC4'])
    ax.set_title(f'Boxplot: {col.split("(")[0][:20]}', fontsize=10, fontweight='bold')

plt.suptitle('Exploratory Data Analysis - Overview', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n✓ EDA Visualizations Complete!")

In [ ]:
# Correlation Analysis
print("\n" + "=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

# Calculate correlation matrix
corr_matrix = X.corr()

# Display top correlations with target
print("\n🔗 Top 10 Features Correlated with Target:")
target_corr = X.corrwith(y).sort_values(ascending=False)
print(target_corr.head(10))

# Visualize correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full correlation heatmap
sns.heatmap(corr_matrix, ax=axes[0], cmap='coolwarm', center=0, square=True, 
            cbar_kws={'label': 'Correlation'}, xticklabels=False)
axes[0].set_title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')

# Target correlation heatmap
target_corr_df = pd.DataFrame(target_corr).sort_values(0, ascending=True)
sns.heatmap(target_corr_df.tail(15), ax=axes[1], annot=True, fmt='.3f', 
            cmap='RdYlGn', center=0, cbar_kws={'label': 'Correlation with Target'})
axes[1].set_title('Top 15 Features vs Target Correlation', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

print("\n✓ Correlation Analysis Complete!")

## 5. Feature Engineering

Create new features, perform scaling/normalization, and select relevant features.

In [ ]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

# Make a copy for feature engineering
X_engineered = X.copy()

# Create polynomial features (example: interaction terms)
print("\n🔧 Creating Feature Engineering:")

# Example: Create ratio features from top correlations
top_features = target_corr.head(3).index.tolist()
print(f"   - Selected Top 3 Features: {[f.split('(')[0][:15] for f in top_features]}")

# Create interaction features
if len(top_features) >= 2:
    X_engineered[f'{top_features[0]}_x_{top_features[1]}'] = X_engineered[top_features[0]] * X_engineered[top_features[1]]
    print(f"   - Created Interaction Feature")

# Feature scaling
print(f"\n📏 Feature Scaling:")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_engineered)
X_scaled = pd.DataFrame(X_scaled, columns=X_engineered.columns)

print(f"   ✓ Applied StandardScaler")
print(f"   Mean of scaled features: {X_scaled.mean().mean():.6f}")
print(f"   Std of scaled features: {X_scaled.std().mean():.6f}")

# Feature selection (select top features)
print(f"\n🎯 Feature Selection (Top 15 Features):")
feature_importance = abs(target_corr).sort_values(ascending=False)
selected_features = feature_importance.head(15).index.tolist()
print(f"   Selected Features: {[f.split('(')[0][:15] for f in selected_features]}")

X_final = X_scaled[selected_features]

print(f"\n✓ Feature Engineering Complete!")
print(f"   Original Feature Count: {X.shape[1]}")
print(f"   Final Feature Count: {X_final.shape[1]}")

## 6. Train-Test Split

Split the dataset into training and testing sets with appropriate ratios.

In [ ]:
print("=" * 80)
print("TRAIN-TEST SPLIT")
print("=" * 80)

# Split the data
test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=test_size, 
    random_state=random_state,
    stratify=y
)

print(f"\n📊 Data Split Summary:")
print(f"   Total Samples: {len(X_final)}")
print(f"   Training Set: {len(X_train)} ({len(X_train)/len(X_final)*100:.1f}%)")
print(f"   Testing Set: {len(X_test)} ({len(X_test)/len(X_final)*100:.1f}%)")

print(f"\n🎯 Class Distribution in Training Set:")
print(f"   {y_train.value_counts().to_dict()}")
print(f"\n🎯 Class Distribution in Testing Set:")
print(f"   {y_test.value_counts().to_dict()}")

print(f"\n✓ Train-Test Split Complete!")

## 7. Model Selection and Training

Train multiple machine learning models and compare their performance.

In [ ]:
print("=" * 80)
print("MODEL TRAINING")
print("=" * 80)

# Dictionary to store models and their results
models = {}
training_results = {}

# Define models to train
model_configs = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)
}

print("\n🚀 Training Models...")
for model_name, model in model_configs.items():
    print(f"\n   Training {model_name}...", end=" ")
    
    # Train the model
    model.fit(X_train, y_train)
    models[model_name] = model
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    training_results[model_name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_predictions': y_train_pred,
        'test_predictions': y_test_pred
    }
    
    print(f"✓ (Train: {train_acc:.4f}, Test: {test_acc:.4f})")

# Display training results comparison
print("\n" + "=" * 80)
print("TRAINING RESULTS COMPARISON")
print("=" * 80)

results_df = pd.DataFrame({
    'Model': list(training_results.keys()),
    'Train Accuracy': [training_results[m]['train_accuracy'] for m in training_results.keys()],
    'Test Accuracy': [training_results[m]['test_accuracy'] for m in training_results.keys()]
})

print("\n📊 Model Performance Summary:")
print(results_df.to_string(index=False))

# Visualize training results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Comparison
x_pos = np.arange(len(results_df))
width = 0.35

axes[0].bar(x_pos - width/2, results_df['Train Accuracy'], width, label='Train', color='#4ECDC4', alpha=0.8)
axes[0].bar(x_pos + width/2, results_df['Test Accuracy'], width, label='Test', color='#FF6B6B', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Training Results Comparison', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Overfitting Analysis
overfitting = results_df['Train Accuracy'] - results_df['Test Accuracy']
colors = ['#FF6B6B' if x > 0.05 else '#4ECDC4' for x in overfitting]
axes[1].barh(results_df['Model'], overfitting, color=colors, alpha=0.8)
axes[1].set_xlabel('Overfitting Gap (Train - Test)')
axes[1].set_title('Overfitting Analysis', fontsize=12, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Model Training Complete!")

## 8. Model Evaluation

Evaluate models using comprehensive metrics and visualizations.

In [ ]:
print("=" * 80)
print("DETAILED MODEL EVALUATION")
print("=" * 80)

# Select best model based on test accuracy
best_model_name = results_df.loc[results_df['Test Accuracy'].idxmax(), 'Model']
best_model = models[best_model_name]
best_test_pred = training_results[best_model_name]['test_predictions']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test Accuracy: {results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0]:.4f}")

# Calculate detailed metrics for best model
print(f"\n📊 Detailed Metrics for {best_model_name}:")
print(f"\n   Accuracy: {accuracy_score(y_test, best_test_pred):.4f}")
print(f"   Precision: {precision_score(y_test, best_test_pred):.4f}")
print(f"   Recall: {recall_score(y_test, best_test_pred):.4f}")
print(f"   F1-Score: {f1_score(y_test, best_test_pred):.4f}")

print(f"\n📋 Classification Report:")
print(classification_report(y_test, best_test_pred, target_names=['Class 0', 'Class 1']))

# Confusion Matrix
cm = confusion_matrix(y_test, best_test_pred)
print(f"\n🔲 Confusion Matrix:")
print(cm)

# Create evaluation visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
axes[0, 0].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# 2. ROC Curve (if probability predictions available)
try:
    y_proba = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    axes[0, 1].plot(fpr, tpr, color='#4ECDC4', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
    axes[0, 1].plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
    axes[0, 1].set_xlim([0.0, 1.0])
    axes[0, 1].set_ylim([0.0, 1.05])
    axes[0, 1].set_xlabel('False Positive Rate')
    axes[0, 1].set_ylabel('True Positive Rate')
    axes[0, 1].set_title(f'ROC Curve - {best_model_name}', fontweight='bold')
    axes[0, 1].legend(loc="lower right")
    axes[0, 1].grid(alpha=0.3)
except:
    axes[0, 1].text(0.5, 0.5, 'ROC Curve\nNot Available', ha='center', va='center')
    axes[0, 1].set_title('ROC Curve', fontweight='bold')

# 3. Metrics Comparison
metrics = {
    'Accuracy': accuracy_score(y_test, best_test_pred),
    'Precision': precision_score(y_test, best_test_pred),
    'Recall': recall_score(y_test, best_test_pred),
    'F1-Score': f1_score(y_test, best_test_pred)
}

axes[1, 0].barh(list(metrics.keys()), list(metrics.values()), color='#4ECDC4', alpha=0.8)
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_title(f'Performance Metrics - {best_model_name}', fontweight='bold')
axes[1, 0].set_xlim([0, 1])
for i, (k, v) in enumerate(metrics.items()):
    axes[1, 0].text(v + 0.02, i, f'{v:.3f}', va='center')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. All Models Metrics Comparison
all_models_metrics = []
for model_name in training_results.keys():
    pred = training_results[model_name]['test_predictions']
    all_models_metrics.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'F1': f1_score(y_test, pred)
    })

metrics_comparison_df = pd.DataFrame(all_models_metrics)
metrics_comparison_df.set_index('Model').plot(kind='bar', ax=axes[1, 1], 
                                               color=['#4ECDC4', '#FF6B6B', '#95E1D3'], alpha=0.8)
axes[1, 1].set_title('All Models - Metrics Comparison', fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_xlabel('Model')
axes[1, 1].legend(title='Metric', loc='lower right')
axes[1, 1].set_ylim([0, 1.1])
axes[1, 1].grid(axis='y', alpha=0.3)
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\n✓ Model Evaluation Complete!")

## 9. Hyperparameter Tuning

Use GridSearchCV to find optimal hyperparameters and improve model performance.

In [ ]:
print("=" * 80)
print("HYPERPARAMETER TUNING")
print("=" * 80)

# Define hyperparameter grids for top models
param_grids = {
    'Logistic Regression': {
        'C': [0.001, 0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10]
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7]
    }
}

tuning_results = {}

print("\n🔧 Performing GridSearchCV for top 3 models...")

for model_name in ['Logistic Regression', 'Random Forest', 'Gradient Boosting']:
    print(f"\n   Tuning {model_name}...", end=" ")
    
    model = model_configs[model_name]
    param_grid = param_grids[model_name]
    
    # Perform grid search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train)
    
    # Store best model and results
    tuning_results[model_name] = {
        'best_model': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'best_cv_score': grid_search.best_score_,
        'test_score': accuracy_score(y_test, grid_search.best_estimator_.predict(X_test))
    }
    
    print(f"✓ (CV Score: {grid_search.best_score_:.4f})")

# Display tuning results
print("\n" + "=" * 80)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 80)

for model_name, results in tuning_results.items():
    print(f"\n📌 {model_name}:")
    print(f"   Best Parameters: {results['best_params']}")
    print(f"   Best CV Score: {results['best_cv_score']:.4f}")
    print(f"   Test Accuracy: {results['test_score']:.4f}")

# Compare original vs tuned models
print("\n" + "=" * 80)
print("ORIGINAL vs TUNED MODELS COMPARISON")
print("=" * 80)

comparison_data = []
for model_name in tuning_results.keys():
    original_test = results_df[results_df['Model'] == model_name]['Test Accuracy'].values[0]
    tuned_test = tuning_results[model_name]['test_score']
    improvement = (tuned_test - original_test) * 100
    
    comparison_data.append({
        'Model': model_name,
        'Original Accuracy': original_test,
        'Tuned Accuracy': tuned_test,
        'Improvement %': improvement
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original vs Tuned
x_pos = np.arange(len(comparison_df))
width = 0.35
axes[0].bar(x_pos - width/2, comparison_df['Original Accuracy'], width, 
           label='Original', color='#FF6B6B', alpha=0.8)
axes[0].bar(x_pos + width/2, comparison_df['Tuned Accuracy'], width, 
           label='Tuned', color='#4ECDC4', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Original vs Tuned Models Performance', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.8, 1.0])

# Improvement percentage
colors = ['#4ECDC4' if x > 0 else '#FF6B6B' for x in comparison_df['Improvement %']]
axes[1].barh(comparison_df['Model'], comparison_df['Improvement %'], color=colors, alpha=0.8)
axes[1].set_xlabel('Improvement %')
axes[1].set_title('Model Performance Improvement', fontsize=12, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1].grid(axis='x', alpha=0.3)

for i, v in enumerate(comparison_df['Improvement %']):
    axes[1].text(v + 0.05, i, f'{v:.2f}%', va='center')

plt.tight_layout()
plt.show()

print("\n✓ Hyperparameter Tuning Complete!")

## 10. Experiment Tracking and Logging

Log experiment parameters, metrics, and results for reproducibility.

In [ ]:
print("=" * 80)
print("EXPERIMENT TRACKING AND LOGGING")
print("=" * 80)

# Create experiment logs directory
os.makedirs('experiment_logs', exist_ok=True)

# Generate experiment summary
experiment_log = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': {
        'total_samples': X_final.shape[0],
        'total_features': X_final.shape[1],
        'train_samples': X_train.shape[0],
        'test_samples': X_test.shape[0],
        'test_size_ratio': test_size
    },
    'preprocessing': {
        'scaling_method': 'StandardScaler',
        'feature_selection': f'Top {X_final.shape[1]} features',
        'train_test_random_state': random_state
    },
    'models': {
        'original_models': {},
        'tuned_models': {}
    },
    'results': {
        'best_model_name': best_model_name,
        'best_model_accuracy': float(results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0])
    }
}

# Add original model results
for idx, row in results_df.iterrows():
    experiment_log['models']['original_models'][row['Model']] = {
        'train_accuracy': float(row['Train Accuracy']),
        'test_accuracy': float(row['Test Accuracy'])
    }

# Add tuned model results
for model_name, results in tuning_results.items():
    experiment_log['models']['tuned_models'][model_name] = {
        'best_parameters': results['best_params'],
        'cv_score': float(results['best_cv_score']),
        'test_accuracy': float(results['test_score'])
    }

# Add comprehensive metrics for best model
best_model_predictions = best_model.predict(X_test)
try:
    best_model_proba = best_model.predict_proba(X_test)[:, 1]
    best_model_auc = roc_auc_score(y_test, best_model_proba)
except:
    best_model_auc = None

experiment_log['results']['best_model_metrics'] = {
    'accuracy': float(accuracy_score(y_test, best_model_predictions)),
    'precision': float(precision_score(y_test, best_model_predictions)),
    'recall': float(recall_score(y_test, best_model_predictions)),
    'f1_score': float(f1_score(y_test, best_model_predictions)),
    'roc_auc': float(best_model_auc) if best_model_auc else 'N/A'
}

print("\n📝 Experiment Summary:")
print(json.dumps(experiment_log, indent=2))

# Save experiment log to file
log_filename = f"experiment_logs/experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(log_filename, 'w') as f:
    json.dump(experiment_log, f, indent=2)

print(f"\n✓ Experiment log saved to: {log_filename}")

# Create a CSV summary of all experiments
experiments_summary = pd.DataFrame([
    {
        'Timestamp': experiment_log['timestamp'],
        'Dataset Size': experiment_log['dataset']['total_samples'],
        'Features Used': experiment_log['dataset']['total_features'],
        'Best Model': experiment_log['results']['best_model_name'],
        'Best Test Accuracy': experiment_log['results']['best_model_metrics']['accuracy'],
        'Best F1-Score': experiment_log['results']['best_model_metrics']['f1_score']
    }
])

csv_filename = 'experiment_logs/experiments_summary.csv'
if os.path.exists(csv_filename):
    experiments_summary = pd.concat([pd.read_csv(csv_filename), experiments_summary], ignore_index=True)

experiments_summary.to_csv(csv_filename, index=False)
print(f"✓ Summary saved to: {csv_filename}")

print("\n✓ Experiment Tracking Complete!")

## 11. Results Comparison and Visualization

Compare results across different models and experiments with comprehensive visualizations.

In [ ]:
print("=" * 80)
print("COMPREHENSIVE RESULTS COMPARISON AND VISUALIZATION")
print("=" * 80)

# Create comprehensive comparison summary
fig = plt.figure(figsize=(20, 14))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Model Accuracy Comparison (All Models)
ax1 = fig.add_subplot(gs[0, :2])
all_models = list(training_results.keys())
test_accs = [training_results[m]['test_accuracy'] for m in all_models]
train_accs = [training_results[m]['train_accuracy'] for m in all_models]

x_pos = np.arange(len(all_models))
width = 0.35
bars1 = ax1.bar(x_pos - width/2, train_accs, width, label='Train Accuracy', color='#4ECDC4', alpha=0.8)
bars2 = ax1.bar(x_pos + width/2, test_accs, width, label='Test Accuracy', color='#FF6B6B', alpha=0.8)

ax1.set_xlabel('Model')
ax1.set_ylabel('Accuracy')
ax1.set_title('All Models - Train vs Test Accuracy', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(all_models, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0.8, 1.0])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

# 2. Original vs Tuned Accuracy
ax2 = fig.add_subplot(gs[0, 2])
model_names_tuned = comparison_df['Model'].tolist()
x_pos_tuned = np.arange(len(model_names_tuned))
width_tuned = 0.35

bars1 = ax2.bar(x_pos_tuned - width_tuned/2, comparison_df['Original Accuracy'], width_tuned,
               label='Original', color='#FF6B6B', alpha=0.8)
bars2 = ax2.bar(x_pos_tuned + width_tuned/2, comparison_df['Tuned Accuracy'], width_tuned,
               label='Tuned', color='#4ECDC4', alpha=0.8)

ax2.set_ylabel('Accuracy')
ax2.set_title('Tuning Impact on Top 3 Models', fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos_tuned)
ax2.set_xticklabels(model_names_tuned, rotation=45, ha='right', fontsize=9)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0.8, 1.0])

# 3. Improvement Percentage
ax3 = fig.add_subplot(gs[1, 0])
colors = ['#4ECDC4' if x > 0 else '#FF6B6B' for x in comparison_df['Improvement %']]
bars = ax3.barh(comparison_df['Model'], comparison_df['Improvement %'], color=colors, alpha=0.8)
ax3.set_xlabel('Improvement %')
ax3.set_title('Hyperparameter Tuning Impact', fontsize=12, fontweight='bold')
ax3.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax3.grid(axis='x', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars, comparison_df['Improvement %'])):
    ax3.text(val + 0.05, i, f'{val:.2f}%', va='center', fontsize=9)

# 4. Best Model Metrics Radar Chart (using bar)
ax4 = fig.add_subplot(gs[1, 1:])
metrics_keys = list(experiment_log['results']['best_model_metrics'].keys())
metrics_values = [experiment_log['results']['best_model_metrics'][k] 
                  for k in metrics_keys if isinstance(experiment_log['results']['best_model_metrics'][k], (int, float))]
metrics_labels = [k.replace('_', ' ').title() for k in metrics_keys 
                  if isinstance(experiment_log['results']['best_model_metrics'][k], (int, float))]

bars = ax4.barh(metrics_labels, metrics_values, color='#4ECDC4', alpha=0.8)
ax4.set_xlabel('Score')
ax4.set_title(f'Best Model Metrics - {best_model_name}', fontsize=12, fontweight='bold')
ax4.set_xlim([0, 1])
ax4.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, metrics_values):
    ax4.text(val + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', fontsize=9)

# 5. Feature Importance (Top 10)
ax5 = fig.add_subplot(gs[2, :2])
if hasattr(best_model, 'feature_importances_'):
    feature_imp = pd.Series(best_model.feature_importances_, index=X_final.columns)
    top_features = feature_imp.nlargest(10)
    
    ax5.barh(range(len(top_features)), top_features.values, color='#95E1D3', alpha=0.8)
    ax5.set_yticks(range(len(top_features)))
    ax5.set_yticklabels(top_features.index)
    ax5.set_xlabel('Importance Score')
    ax5.set_title('Top 10 Feature Importance', fontsize=12, fontweight='bold')
    ax5.grid(axis='x', alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'Feature Importance\nNot Available', ha='center', va='center',
            fontsize=11, transform=ax5.transAxes)
    ax5.set_title('Top 10 Feature Importance', fontsize=12, fontweight='bold')

# 6. Experiment Summary Table
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')

summary_text = f"""
EXPERIMENT SUMMARY
════════════════════════
Dataset: {X_final.shape[0]} samples
Features: {X_final.shape[1]}

BEST MODEL: {best_model_name}
────────────────────────
Accuracy: {experiment_log['results']['best_model_metrics']['accuracy']:.4f}
Precision: {experiment_log['results']['best_model_metrics']['precision']:.4f}
Recall: {experiment_log['results']['best_model_metrics']['recall']:.4f}
F1-Score: {experiment_log['results']['best_model_metrics']['f1_score']:.4f}

IMPROVEMENT
────────────────────────
Tuned vs Original: {comparison_df[comparison_df['Model']==best_model_name]['Improvement %'].values[0] if best_model_name in comparison_df['Model'].values else 'N/A'}%
"""

ax6.text(0.05, 0.5, summary_text, fontsize=10, family='monospace',
        verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Comprehensive Machine Learning Pipeline Results', fontsize=16, fontweight='bold', y=0.995)
plt.show()

print("\n✓ Comprehensive Results Visualization Complete!")

In [ ]:
# Final Summary Report
print("\n" + "=" * 80)
print("FINAL SUMMARY REPORT")
print("=" * 80)

summary_report = f"""
🎯 PROJECT COMPLETION SUMMARY
══════════════════════════════════════════════════════════════════════════════

📊 DATA PIPELINE
  • Total Samples: {X_final.shape[0]}
  • Total Features: {X_final.shape[1]}
  • Training Samples: {X_train.shape[0]} ({X_train.shape[0]/X_final.shape[0]*100:.1f}%)
  • Testing Samples: {X_test.shape[0]} ({X_test.shape[0]/X_final.shape[0]*100:.1f}%)
  • Class Distribution: Stratified (Balanced)

🔧 PREPROCESSING & ENGINEERING
  • Missing Values: Handled
  • Scaling Method: StandardScaler
  • Feature Selection: Top {X_final.shape[1]} features selected
  • Handling: Complete

🤖 MODELS TRAINED
  • Logistic Regression: {training_results['Logistic Regression']['test_accuracy']:.4f}
  • Random Forest: {training_results['Random Forest']['test_accuracy']:.4f}
  • Gradient Boosting: {training_results['Gradient Boosting']['test_accuracy']:.4f}
  • SVM: {training_results['SVM']['test_accuracy']:.4f}

🏆 BEST MODEL: {best_model_name}
  • Original Test Accuracy: {results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0]:.4f}
  • Tuned Test Accuracy: {tuning_results[best_model_name]['test_score']:.4f}
  • Improvement: {(tuning_results[best_model_name]['test_score'] - results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0])*100:.2f}%

📈 FINAL METRICS (Best Model on Test Set)
  • Accuracy: {experiment_log['results']['best_model_metrics']['accuracy']:.4f}
  • Precision: {experiment_log['results']['best_model_metrics']['precision']:.4f}
  • Recall: {experiment_log['results']['best_model_metrics']['recall']:.4f}
  • F1-Score: {experiment_log['results']['best_model_metrics']['f1_score']:.4f}
  • ROC-AUC: {experiment_log['results']['best_model_metrics']['roc_auc'] if isinstance(experiment_log['results']['best_model_metrics']['roc_auc'], (int, float)) else 'N/A'}

💾 ARTIFACTS SAVED
  • Experiment Log: experiment_logs/experiment_*.json
  • Summary CSV: experiment_logs/experiments_summary.csv
  • Timestamp: {experiment_log['timestamp']}

✅ PIPELINE STATUS: COMPLETE
  All steps from EDA to hyperparameter tuning have been successfully executed!
  
════════════════════════════════════════════════════════════════════════════════
"""

print(summary_report)

# Save summary report
report_filename = f"experiment_logs/summary_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_filename, 'w') as f:
    f.write(summary_report)

print(f"\n✓ Final report saved to: {report_filename}")
print("✓ All notebooks cells executed successfully!")

In [ ]:
# Final Summary Report
print("\n" + "=" * 80)
print("FINAL SUMMARY REPORT")
print("=" * 80)

summary_report = f"""
🎯 PROJECT COMPLETION SUMMARY
══════════════════════════════════════════════════════════════════════════════

📊 DATA PIPELINE
  • Total Samples: {X_final.shape[0]}
  • Total Features: {X_final.shape[1]}
  • Training Samples: {X_train.shape[0]} ({X_train.shape[0]/X_final.shape[0]*100:.1f}%)
  • Testing Samples: {X_test.shape[0]} ({X_test.shape[0]/X_final.shape[0]*100:.1f}%)
  • Class Distribution: Stratified (Balanced)

🔧 PREPROCESSING & ENGINEERING
  • Missing Values: Handled
  • Scaling Method: StandardScaler
  • Feature Selection: Top {X_final.shape[1]} features selected
  • Handling: Complete

🤖 MODELS TRAINED
  • Logistic Regression: {training_results['Logistic Regression']['test_accuracy']:.4f}
  • Random Forest: {training_results['Random Forest']['test_accuracy']:.4f}
  • Gradient Boosting: {training_results['Gradient Boosting']['test_accuracy']:.4f}
  • SVM: {training_results['SVM']['test_accuracy']:.4f}

🏆 BEST MODEL: {best_model_name}
  • Original Test Accuracy: {results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0]:.4f}
  • Tuned Test Accuracy: {tuning_results[best_model_name]['test_score']:.4f}
  • Improvement: {(tuning_results[best_model_name]['test_score'] - results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0])*100:.2f}%

📈 FINAL METRICS (Best Model on Test Set)
  • Accuracy: {experiment_log['results']['best_model_metrics']['accuracy']:.4f}
  • Precision: {experiment_log['results']['best_model_metrics']['precision']:.4f}
  • Recall: {experiment_log['results']['best_model_metrics']['recall']:.4f}
  • F1-Score: {experiment_log['results']['best_model_metrics']['f1_score']:.4f}
  • ROC-AUC: {experiment_log['results']['best_model_metrics']['roc_auc'] if isinstance(experiment_log['results']['best_model_metrics']['roc_auc'], (int, float)) else 'N/A'}

💾 ARTIFACTS SAVED
  • Experiment Log: experiment_logs/experiment_*.json
  • Summary CSV: experiment_logs/experiments_summary.csv
  • Timestamp: {experiment_log['timestamp']}

✅ PIPELINE STATUS: COMPLETE
  All steps from EDA to hyperparameter tuning have been successfully executed!
  
════════════════════════════════════════════════════════════════════════════════
"""

print(summary_report)

# Save summary report
report_filename = f"experiment_logs/summary_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_filename, 'w') as f:
    f.write(summary_report)

print(f"\n✓ Final report saved to: {report_filename}")
print("✓ All notebooks cells executed successfully!")

In [ ]:
# Final Summary Report
print("\n" + "=" * 80)
print("FINAL SUMMARY REPORT")
print("=" * 80)

summary_report = f"""
🎯 PROJECT COMPLETION SUMMARY
══════════════════════════════════════════════════════════════════════════════

📊 DATA PIPELINE
  • Total Samples: {X_final.shape[0]}
  • Total Features: {X_final.shape[1]}
  • Training Samples: {X_train.shape[0]} ({X_train.shape[0]/X_final.shape[0]*100:.1f}%)
  • Testing Samples: {X_test.shape[0]} ({X_test.shape[0]/X_final.shape[0]*100:.1f}%)
  • Class Distribution: Stratified (Balanced)

🔧 PREPROCESSING & ENGINEERING
  • Missing Values: Handled
  • Scaling Method: StandardScaler
  • Feature Selection: Top {X_final.shape[1]} features selected
  • Handling: Complete

🤖 MODELS TRAINED
  • Logistic Regression: {training_results['Logistic Regression']['test_accuracy']:.4f}
  • Random Forest: {training_results['Random Forest']['test_accuracy']:.4f}
  • Gradient Boosting: {training_results['Gradient Boosting']['test_accuracy']:.4f}
  • SVM: {training_results['SVM']['test_accuracy']:.4f}

🏆 BEST MODEL: {best_model_name}
  • Original Test Accuracy: {results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0]:.4f}
  • Tuned Test Accuracy: {tuning_results[best_model_name]['test_score']:.4f}
  • Improvement: {(tuning_results[best_model_name]['test_score'] - results_df[results_df['Model']==best_model_name]['Test Accuracy'].values[0])*100:.2f}%

📈 FINAL METRICS (Best Model on Test Set)
  • Accuracy: {experiment_log['results']['best_model_metrics']['accuracy']:.4f}
  • Precision: {experiment_log['results']['best_model_metrics']['precision']:.4f}
  • Recall: {experiment_log['results']['best_model_metrics']['recall']:.4f}
  • F1-Score: {experiment_log['results']['best_model_metrics']['f1_score']:.4f}
  • ROC-AUC: {experiment_log['results']['best_model_metrics']['roc_auc'] if isinstance(experiment_log['results']['best_model_metrics']['roc_auc'], (int, float)) else 'N/A'}

💾 ARTIFACTS SAVED
  • Experiment Log: experiment_logs/experiment_*.json
  • Summary CSV: experiment_logs/experiments_summary.csv
  • Timestamp: {experiment_log['timestamp']}

✅ PIPELINE STATUS: COMPLETE
  All steps from EDA to hyperparameter tuning have been successfully executed!
  
════════════════════════════════════════════════════════════════════════════════
"""

print(summary_report)

# Save summary report
report_filename = f"experiment_logs/summary_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_filename, 'w') as f:
    f.write(summary_report)

print(f"\n✓ Final report saved to: {report_filename}")
print("✓ All notebooks cells executed successfully!")